In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

# Step 1: Indexing (Document Ingestion)

In [40]:
video_id = "VDBncKw5v_o" # From url

try:
  transcript_obj = YouTubeTranscriptApi().fetch(video_id=video_id, languages=["en"])
except TranscriptsDisabled:
  print("Transcripts are disabled for this video.")

In [41]:
print(transcript_obj)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=0.98, duration=7.02), FetchedTranscriptSnippet(text='Once upon a time in feudal Japan, there', start=5.44, duration=5.36), FetchedTranscriptSnippet(text='lived a young samurai named Yuki who', start=8.0, duration=4.96), FetchedTranscriptSnippet(text='dreamed of becoming the greatest warrior', start=10.8, duration=4.799), FetchedTranscriptSnippet(text='in the land. He would spend hours', start=12.96, duration=4.88), FetchedTranscriptSnippet(text='speaking about his plans, reading', start=15.599, duration=5.121), FetchedTranscriptSnippet(text='stories of famous battles and imagining', start=17.84, duration=6.32), FetchedTranscriptSnippet(text='himself winning great victories. Yuki', start=20.72, duration=6.0), FetchedTranscriptSnippet(text='dreamed of mastering many weapons,', start=24.16, duration=5.039), FetchedTranscriptSnippet(text='learning advanced strategies, and', start=26.72, duration=5.04), FetchedTransc

In [42]:
# for transcript in transcript_obj:
#   print(transcript.text)

transcript = " ".join(chunk.text for chunk in transcript_obj)
print(transcript)

[Music] Once upon a time in feudal Japan, there lived a young samurai named Yuki who dreamed of becoming the greatest warrior in the land. He would spend hours speaking about his plans, reading stories of famous battles and imagining himself winning great victories. Yuki dreamed of mastering many weapons, learning advanced strategies, and earning honor for his clan. He truly believed that having such strong ambition would naturally lead him to greatness. Yet, when the time came to train, Yuki always found excuses to avoid the hard work. He told himself the weather was not right or that he was not feeling well, or that he needed to plan more carefully before beginning. Years passed and while Yuki's dreams grew larger, his skills remained small. Younger warriors who trained everyday soon surpassed him and Yuki came to realize a painful truth. His dreams had not lifted him higher. They had made him weaker. His ambition without action had left him frustrated, unprepared, and full of regret

In [43]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [44]:
len(chunks)

7

In [45]:
chunks[0]

Document(metadata={}, page_content="[Music] Once upon a time in feudal Japan, there lived a young samurai named Yuki who dreamed of becoming the greatest warrior in the land. He would spend hours speaking about his plans, reading stories of famous battles and imagining himself winning great victories. Yuki dreamed of mastering many weapons, learning advanced strategies, and earning honor for his clan. He truly believed that having such strong ambition would naturally lead him to greatness. Yet, when the time came to train, Yuki always found excuses to avoid the hard work. He told himself the weather was not right or that he was not feeling well, or that he needed to plan more carefully before beginning. Years passed and while Yuki's dreams grew larger, his skills remained small. Younger warriors who trained everyday soon surpassed him and Yuki came to realize a painful truth. His dreams had not lifted him higher. They had made him weaker. His ambition without action had left him frustr

In [46]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = Chroma.from_documents(chunks, embeddings)

# Step 2: Retrieval

In [47]:
retriever = vector_store.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 4}
)

In [48]:
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7433fc3741d0>, search_kwargs={'k': 4})

In [49]:
retriever.invoke("What did yuki dream about?")

[Document(id='d4f48d27-a8b9-45b5-a658-ea581bb125f2', metadata={}, page_content="[Music] Once upon a time in feudal Japan, there lived a young samurai named Yuki who dreamed of becoming the greatest warrior in the land. He would spend hours speaking about his plans, reading stories of famous battles and imagining himself winning great victories. Yuki dreamed of mastering many weapons, learning advanced strategies, and earning honor for his clan. He truly believed that having such strong ambition would naturally lead him to greatness. Yet, when the time came to train, Yuki always found excuses to avoid the hard work. He told himself the weather was not right or that he was not feeling well, or that he needed to plan more carefully before beginning. Years passed and while Yuki's dreams grew larger, his skills remained small. Younger warriors who trained everyday soon surpassed him and Yuki came to realize a painful truth. His dreams had not lifted him higher. They had made him weaker. His

# Step 3: Augmentation

In [50]:
GEMINI_MODEL = "gemma-3-27b-it"
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.2)

In [51]:
prompt = PromptTemplate(
  template="""
  You are a helpful assistant. Answer only from the provided transcript context. If context is insufficient, just say you don't know.
  
  {context}
  Question: {question}
  
  """,
  input_variables=["context", "question"]
)

In [66]:
question = "What did yuki dream about?"
question = "tell me about mousashi"
question = "is there any discussion about langchain?"
retrieved_docs = retriever.invoke(question)

In [67]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [68]:
context_text

'the way of small improvements. The mountain is climbed one step at a time, they said, not by staring at its peak and wishing to be there. Even if the steps seem small, they lead to strength because they are repeated. Even if the progress seems slow, it builds into mastery because it does not stop. Imagine a samurai student who feels too lazy to train yet chooses to practice a single sword form each morning. His friends laugh at the tiny effort and tell him it is useless. But he continues. Over time, the movement becomes automatic. His body grows stronger, his technique sharper, his discipline deeper. Months later, he realizes he has built a foundation that allows him to learn advanced skills with ease while his friends who waited for motivation remain at the beginning. In this way, the smallest step can carry greater power than the grandest plan. The lesson is clear. Preparation without practice is simply disguised procrastination. Talking without doing creates the illusion of\n\nthe 

In [69]:
final_prompt = prompt.invoke(
  {
    "context": context_text,
    "question": question
  }
)

final_prompt


StringPromptValue(text='\n  You are a helpful assistant. Answer only from the provided transcript context. If context is insufficient, just say you don\'t know.\n\n  the way of small improvements. The mountain is climbed one step at a time, they said, not by staring at its peak and wishing to be there. Even if the steps seem small, they lead to strength because they are repeated. Even if the progress seems slow, it builds into mastery because it does not stop. Imagine a samurai student who feels too lazy to train yet chooses to practice a single sword form each morning. His friends laugh at the tiny effort and tell him it is useless. But he continues. Over time, the movement becomes automatic. His body grows stronger, his technique sharper, his discipline deeper. Months later, he realizes he has built a foundation that allows him to learn advanced skills with ease while his friends who waited for motivation remain at the beginning. In this way, the smallest step can carry greater power

# Step 4: Generation

In [70]:
answer = llm.invoke(final_prompt)

answer.content

"I don't know. The provided text does not contain any information about langchain."

## Building a chain

In [71]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [72]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [73]:
prallel_chain = RunnableParallel(
  {
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
  }
)

In [74]:
prallel_chain.invoke("what is yuki's dream?")

{'context': "[Music] Once upon a time in feudal Japan, there lived a young samurai named Yuki who dreamed of becoming the greatest warrior in the land. He would spend hours speaking about his plans, reading stories of famous battles and imagining himself winning great victories. Yuki dreamed of mastering many weapons, learning advanced strategies, and earning honor for his clan. He truly believed that having such strong ambition would naturally lead him to greatness. Yet, when the time came to train, Yuki always found excuses to avoid the hard work. He told himself the weather was not right or that he was not feeling well, or that he needed to plan more carefully before beginning. Years passed and while Yuki's dreams grew larger, his skills remained small. Younger warriors who trained everyday soon surpassed him and Yuki came to realize a painful truth. His dreams had not lifted him higher. They had made him weaker. His ambition without action had left him frustrated, unprepared, and f

In [75]:
parser = StrOutputParser()

In [77]:
main_chain = prallel_chain | prompt | llm | parser

In [78]:
main_chain.invoke("Summarize this video")

'This video contrasts the approach of a young samurai, Yuki, who dreamed of greatness but lacked discipline, with the wisdom of samurai tradition and swordsman Miiamoto Mousashi. Yuki’s ambition without action led to frustration and being surpassed by others. The video emphasizes that true strength isn’t about *feeling* motivated, but about acting *despite* a lack of motivation, through small daily habits and commitments. It highlights that dreams without action are wasted time, and greatness is built through consistent, humble practice – not waiting for the perfect moment or relying on fleeting feelings. The core message is to focus on *doing* rather than just *wanting*.'